# Stage 3 on Colab: CatBoost tuning and final models on GPU / Этап 3 в Colab: подбор и итоговые модели CatBoost на GPU

**EN.** This notebook runs the long CatBoost part of stage 3 on a Colab GPU: 3 Optuna studies × 40 trials (main model on the 18 selected features; the static and the static + dynamic models of hypothesis H3) and the final 5-fold models of every CatBoost variant. The task is Y1 only. The code is the project code (`scripts/tune.py`, `scripts/fit_models.py`) with `XAI_CATBOOST_DEVICE=GPU`. The bundle contains only train and validation; **test is not in it**.

Before running: (1) Runtime → Change runtime type → **GPU (T4)**; (2) upload `colab_bundle.zip` to Google Drive into the **new** folder `AITU/xai_stage3_v2` (not the old `xai_stage3`: it holds the studies of the previous run under the same names, and they would be taken as finished). Then Runtime → Run all. If the session stops, run all again: finished trials and models are on Drive and are skipped.

At the end download the Drive folder `AITU/xai_stage3_v2/models` (it contains `optuna.db` and one folder per model).

**RU.** Ноутбук выполняет долгую часть этапа 3 с CatBoost на GPU в Colab: 3 подбора Optuna по 40 попыток (основная модель на 18 отобранных признаках; статическая и статическая + динамическая модели для гипотезы H3) и итоговые 5-фолдовые модели всех вариантов CatBoost. Задача только Y1. Код — код проекта (`scripts/tune.py`, `scripts/fit_models.py`) с `XAI_CATBOOST_DEVICE=GPU`. В пакете только train и validation; **test в него не входит**.

Перед запуском: (1) Среда выполнения → Сменить среду выполнения → **GPU (T4)**; (2) загрузите `colab_bundle.zip` на Google Диск в **новую** папку `AITU/xai_stage3_v2` (не в старую `xai_stage3`: там лежат подборы прошлого запуска с теми же именами, и они будут приняты за готовые). Затем Среда выполнения → Выполнить все. Если сессия оборвётся, снова «Выполнить все»: готовые попытки и модели лежат на Диске и пропускаются.

В конце скачайте папку Диска `AITU/xai_stage3_v2/models` (в ней `optuna.db` и по папке на каждую модель).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, subprocess, zipfile
DRIVE = "/content/drive/MyDrive/AITU/xai_stage3_v2"
ROOT = "/content/xai"
MODELS_DRIVE = f"{DRIVE}/models"
os.makedirs(MODELS_DRIVE, exist_ok=True)
if not os.path.exists(f"{ROOT}/src"):
    with zipfile.ZipFile(f"{DRIVE}/colab_bundle.zip") as z:
        z.extractall(ROOT)
# fitted models go straight to Drive; the Optuna journal stays on the local disk and is copied
# to Drive after every trial (SQLite on Drive does not lock reliably)
models_local = f"{ROOT}/data/processed/models"
if os.path.isdir(models_local) and not os.path.islink(models_local):
    shutil.rmtree(models_local)
if not os.path.exists(models_local):
    os.symlink(MODELS_DRIVE, models_local)
if os.path.exists(f"{MODELS_DRIVE}/optuna.db") and not os.path.exists("/content/optuna.db"):
    shutil.copy(f"{MODELS_DRIVE}/optuna.db", "/content/optuna.db")
os.environ.update({"XAI_CATBOOST_DEVICE": "GPU", "XAI_OPTUNA_DB": "/content/optuna.db",
                   "XAI_BACKUP_DIR": MODELS_DRIVE, "PYTHONPATH": f"{ROOT}/src"})
print(os.listdir(ROOT), os.listdir(MODELS_DRIVE))

In [ ]:
!pip -q install catboost==1.2.10 optuna==5.0.0 polars==2.0.0 pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
def run(args):
    # stream the script's output into the notebook
    p = subprocess.Popen(["python", *args], cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not line.startswith("[I "):
            print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"{args} failed")

### 1. Tuning / Подбор (3 × 40 trials)

In [ ]:
for fset in ["main", "h3_static", "h3_full"]:
    run(["scripts/tune.py", "catboost", "y1", fset])

### 2. Final fold models / Итоговые фолдовые модели

In [ ]:
CATBOOST_VARIANTS = ["m1_y1", "h3_static_y1", "h3_full_y1", "m1_y1_noomega", "m1_y1_linear", "m1_y1_r01",
                     "m1_y1_noprepaid"]
run(["scripts/fit_models.py", *CATBOOST_VARIANTS])
shutil.copy("/content/optuna.db", MODELS_DRIVE)
sorted(os.listdir(MODELS_DRIVE))